In [ ]:

# ===============================================
# In This file:
# The Single-generator R-GAN framework is implemented on the CIFAR-10 dataset for Untargeted Attacks, according the R-GAN paper.
# The robustness (accuracy) for classifiers of R-net, C-net, and T-net are evaluated against attacks generated by G-net, FGSM, PGD, and C&W.
# ===============================================


In [ ]:
# === Change This Path to your actual desktop path ===
dataset_path = r"C:\Users\Administrator\Desktop\R-GAN\Dataset\cifar-10-python"

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:
# Define transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])


# Load CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=True,
    transform=transform_train,
    download=False
)

test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_path,
    train=False,
    transform=transform_test,
    download=False
)

batch_size = 128
num_workers = 2 if torch.cuda.is_available() else 0

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(" CIFAR-10 loaded successfully!")
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# CIFAR-10 normalization
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

def denormalize(imgs_norm):
    """imgs_norm: normalized (mean/std) -> returns imgs in [0,1] pixel space"""
    return imgs_norm * CIFAR10_STD + CIFAR10_MEAN

def renormalize(imgs_01):
    """imgs_01 in [0,1] -> normalized"""
    return (imgs_01 - CIFAR10_MEAN) / CIFAR10_STD

# L_inf epsilon in pixel space
eps = 8.0 / 255.0
print("eps (L_inf):", eps)


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the classifiers f-net (C-net) and R-net is defined as Wide ResNet-34.
# The pretrained classifier C-net described in the paper is referred to as f-net in the code (f-net in code = C-net in the paper).
# ===============================================


class BasicBlockWide(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1, dropRate=0.0):
        super(BasicBlockWide, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.shortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False) or None

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.shortcut is None else self.shortcut(x), out)


class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        layers = []
        for i in range(nb_layers):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes,
                                i == 0 and stride or 1, dropRate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet_CIFAR(nn.Module):
    def __init__(self, depth=34, widen_factor=10, num_classes=10, dropRate=0.0):
        super(WideResNet_CIFAR, self).__init__()
        nChannels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert ((depth - 4) % 6 == 0), 'Depth should be 6n+4'
        n = (depth - 4) // 6
        block = BasicBlockWide

        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(nChannels[3], num_classes)

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        return self.fc(out)


def WideResNet34_10_CIFAR():
    """Return WRN-34-10 (depth=34, widen_factor=10)"""
    return WideResNet_CIFAR(depth=34, widen_factor=10, num_classes=10)


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the generator is defined.
# The generator is denoted as G in the code and as G-net in the paper (G in the code = G-net in the paper).
# ===============================================


class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.BatchNorm2d, use_dropout=False, use_bias=False):
        super().__init__()
        conv_block = [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim),
            nn.ReLU(True)
        ]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]
        conv_block += [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim)
        ]
        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        return x + self.conv_block(x)

class Generator(nn.Module):
    def __init__(self, gen_input_nc=3, image_nc=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(gen_input_nc, 32, 3, 1, 1), nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, 2, 1),           nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, 2, 1),          nn.InstanceNorm2d(128), nn.ReLU(True),
        )
        self.bottleneck = nn.Sequential(ResnetBlock(128), ResnetBlock(128), ResnetBlock(128))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),  nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, image_nc, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, x_01):
        # x_01 in [0,1]
        return self.decoder(self.bottleneck(self.encoder(x_01)))


In [ ]:
# ===============================================
# In This Cell:
# The classifier f-net (C-net) is trained using Wide ResNet-34.
# ===============================================


f_net = WideResNet34_10_CIFAR().to(device)
criterion = nn.CrossEntropyLoss()
optimizer_f = optim.SGD(f_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_f = optim.lr_scheduler.MultiStepLR(optimizer_f, milestones=[80, 120], gamma=0.1)

epochs_f = 150
print("Training f_net (WideResNet-34-10) for", epochs_f, "epochs...")

for epoch in range(1, epochs_f + 1):
    f_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm, labels = imgs_norm.to(device), labels.to(device)
        optimizer_f.zero_grad()
        logits = f_net(imgs_norm)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_f.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total
    train_loss = running_loss / len(train_dataloader)

    if (epoch == 1 or epoch % 10 == 0 or epoch == epochs_f):
        f_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm, labels_t = imgs_t_norm.to(device), labels_t.to(device)
                logits_t = f_net(imgs_t_norm)
                loss_t = criterion(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)
        test_acc = 100.0 * correct_test / total_test
        test_loss = test_loss / len(test_dataloader)
        print(f"[f_train] Epoch {epoch}/{epochs_f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")
    scheduler_f.step()

# Freeze and set eval mode
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False


In [ ]:
# ===============================================
# In This Cell:
# The R-GAN Framework is trained.
# R-net and G(G-net) are trained together competitively in the R-GAN Framework.
# The R-GAN Framework in this cell and file is implemented for the Untargeted Attack scenario.
# ===============================================


# define epsislon
eps = 8.0 / 255.0

# Build R_net, G and optimizers
R_net = WideResNet34_10_CIFAR().to(device)
optimizer_R = optim.SGD(R_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# --- Scheduler for R_net ---
scheduler_R = optim.lr_scheduler.MultiStepLR(optimizer_R, milestones=[80, 120], gamma=0.1)

G = Generator(gen_input_nc=3, image_nc=3).to(device)
optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-3)

# Hyperparams
lambda_f = 5.0
lambda_R = 5.0
lambda_pert = 1.0
margin = 0.0

num_epochs = 150    
print_every = 1

def hinge_adv_loss_logits(logits, labels, margin=0.0):
    B = logits.shape[0]
    true_logits = logits[torch.arange(B), labels]
    other_logits = logits.clone()
    other_logits[torch.arange(B), labels] = -1e9
    max_other, _ = other_logits.max(dim=1)
    loss = torch.clamp(true_logits - max_other + margin, min=0.0)
    return loss.mean()

# f_net is frozen & in evaluation mode
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False

# Training loop
for epoch in range(1, num_epochs + 1):
    G.train(); R_net.train()
    total_loss_R = 0.0
    total_loss_G = 0.0
    total_loss_f = 0.0
    total_loss_pert = 0.0
    batches = 0

    for imgs_norm, labels in train_dataloader:
        imgs_norm, labels = imgs_norm.to(device), labels.to(device)
        batches += 1

        imgs_pixel = denormalize(imgs_norm)   # imgs_norm pixel [0,1] for G

        # -------------------------
        # 1) R-update: train R on clean + adv (G frozen)
        # -------------------------
        G.eval()
        with torch.no_grad():
            delta = G(imgs_pixel)                 
            delta = torch.clamp(delta, -eps, eps)  
            adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
            adv_norm = renormalize(adv_pixel)      # normalized for classifier input

        R_net.train()
        logits_clean = R_net(imgs_norm)   # imgs_norm already normalized by dataloader transform
        logits_adv   = R_net(adv_norm)
        loss_R = F.cross_entropy(logits_clean, labels) + F.cross_entropy(logits_adv, labels)

        optimizer_R.zero_grad()
        loss_R.backward()
        optimizer_R.step()
        total_loss_R += loss_R.item()

        # -------------------------
        # 2) G-update: train G to fool f and R (freeze R params)
        # -------------------------
        G.train()
        R_net.eval()
        for p in R_net.parameters():
            p.requires_grad = False

        delta = G(imgs_pixel)
        delta = torch.clamp(delta, -eps, eps)
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)

        # loss vs pretrained f (encourage misclassification)
        logits_f = f_net(adv_norm)
        loss_f_adv = hinge_adv_loss_logits(logits_f, labels, margin=margin)

        # loss vs R (encourage misclassification of R during G update)
        logits_R_adv = R_net(adv_norm)
        loss_R_adv = hinge_adv_loss_logits(logits_R_adv, labels, margin=margin)

        # perturbation regularizer
        loss_pert = torch.mean(torch.norm(delta.view(delta.shape[0], -1), dim=1, p=2))

        loss_G = lambda_f * loss_f_adv + lambda_R * loss_R_adv + lambda_pert * loss_pert

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()
        total_loss_G += loss_G.item()
        total_loss_f += loss_f_adv.item()
        total_loss_pert += loss_pert.item()

        # unfreeze R params for next iteration
        for p in R_net.parameters():
            p.requires_grad = True

    # Epoch summaries
    avg_R = total_loss_R / (batches if batches else 1)
    avg_G = total_loss_G / (batches if batches else 1)
    avg_f = total_loss_f / (batches if batches else 1)
    avg_pert = total_loss_pert / (batches if batches else 1)

    if epoch % print_every == 0:
        print(f"[Epoch {epoch}/{num_epochs}] Loss_R: {avg_R:.4f} | Loss_G: {avg_G:.4f} | Loss_f: {avg_f:.4f} | Loss_pert(L2): {avg_pert:.4f}")

    # Evaluation on test set each epoch (clean and adv by current G)
    G.eval(); R_net.eval(); f_net.eval()
    correct_clean_R = correct_adv_R = 0
    correct_clean_f = correct_adv_f = 0
    total_test = 0
    with torch.no_grad():
        for imgs_t_norm, labels_t in test_dataloader:
            imgs_t_norm, labels_t = imgs_t_norm.to(device), labels_t.to(device)
            total_test += labels_t.size(0)

            preds_R_clean = R_net(imgs_t_norm).argmax(dim=1)
            preds_f_clean = f_net(imgs_t_norm).argmax(dim=1)
            correct_clean_R += (preds_R_clean == labels_t).sum().item()
            correct_clean_f += (preds_f_clean == labels_t).sum().item()

            imgs_t_pixel = denormalize(imgs_t_norm)
            delta_t = torch.clamp(G(imgs_t_pixel), -eps, eps)
            adv_t_pixel = torch.clamp(imgs_t_pixel + delta_t, 0.0, 1.0)
            adv_t_norm = renormalize(adv_t_pixel)

            preds_R_adv = R_net(adv_t_norm).argmax(dim=1)
            preds_f_adv = f_net(adv_t_norm).argmax(dim=1)
            correct_adv_R += (preds_R_adv == labels_t).sum().item()
            correct_adv_f += (preds_f_adv == labels_t).sum().item()

    print(f" Eval - R clean acc: {100*correct_clean_R/total_test:.2f}% | R adv acc: {100*correct_adv_R/total_test:.2f}%")
    print(f"      f clean acc: {100*correct_clean_f/total_test:.2f}% | f adv acc: {100*correct_adv_f/total_test:.2f}%")

    # step scheduler for R_net every epoch
    scheduler_R.step()




In [ ]:
# ===============================================
# In the three cells below:
# T-net is trined using Model ResNet-18.
# To train T-net with Model ResNet-32 or Model Wide ResNet-34, as mentioned in the R-GAN Paper, set T-net according to the architecture of these models.
# The Architecture of Model ResNet-32 or Wide ResNet-34 is defined in the "Architecture of Models" file.
# ===============================================


In [ ]:
# Shuffle the train dataset
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
print("The Train Dataset is shuffled in order difference with Victim Model")

In [ ]:
# ===============================================
# In This Cell:
# The Architecture of ResNet-18 (T-net) is defined (CIFAR-10 version).
# ===============================================


# Basic residual block
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ResNet-18 adapted for CIFAR-10
class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_CIFAR, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)  # global avg pool
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18_CIFAR():
    return ResNet_CIFAR(BasicBlock, [2,2,2,2])


In [ ]:

# ===============================================
# In This Cell:
# The classifier of T-net is trained.
# ===============================================

import torch
import torch.nn as nn
import torch.optim as optim

# build T_net
T_net = ResNet18_CIFAR().to(device)

# loss / optimizer / scheduler 
criterion_t = nn.CrossEntropyLoss()
optimizer_t = optim.SGD(T_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_t = optim.lr_scheduler.MultiStepLR(optimizer_t, milestones=[80, 120], gamma=0.1)

epochs_t = 150
print("Training T_net (ResNet18) for", epochs_t, "epochs...")

for epoch in range(1, epochs_t + 1):
    # ---- Train ----
    T_net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs_norm, labels in train_dataloader:
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)

        optimizer_t.zero_grad()
        logits = T_net(imgs_norm)           
        loss = criterion_t(logits, labels)
        loss.backward()
        optimizer_t.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    train_loss = running_loss / (len(train_dataloader) if len(train_dataloader)>0 else 1)

    # ---- Test / eval periodically ----
    if (epoch == 1 or epoch % 10 == 0 or epoch == epochs_t):
        T_net.eval()
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        with torch.no_grad():
            for imgs_t_norm, labels_t in test_dataloader:
                imgs_t_norm = imgs_t_norm.to(device)
                labels_t = labels_t.to(device)
                logits_t = T_net(imgs_t_norm)
                loss_t = criterion_t(logits_t, labels_t)
                test_loss += loss_t.item()
                preds_t = logits_t.argmax(dim=1)
                correct_test += (preds_t == labels_t).sum().item()
                total_test += labels_t.size(0)

        test_acc = 100.0 * correct_test / total_test if total_test>0 else 0.0
        test_loss = test_loss / (len(test_dataloader) if len(test_dataloader)>0 else 1)
        print(f"[T_train] Epoch {epoch}/{epochs_t} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

    scheduler_t.step()

# Freeze T_net and set eval 
T_net.eval()
for p in T_net.parameters():
    p.requires_grad = False

print("Finished training T_net; model set to eval and frozen.")


In [ ]:

# ================================================
# In This Cell: 
# The accuracy of T_net, R_net, and f_net for perturbed data by generator G (G-net attacker) and clean data is evaluated.
# In this file, the generator G-net produce Untargeted Attacks.
# ================================================



import torch
from tqdm import tqdm
import torch.nn.functional as F

# ensure models are in eval mode
G.eval()
T_net.eval()
R_net.eval()
f_net.eval()

# pixel-space epsilon used in training / G clamping 
try:
    eps  # if defined previously
except NameError:
    eps = 8.0 / 255.0

total = 0
correct_clean_T = 0
correct_adv_T = 0

correct_clean_R = 0
correct_adv_R = 0
correct_clean_f = 0
correct_adv_f = 0

with torch.no_grad():
    for imgs_norm, labels in tqdm(test_dataloader, desc="Eval T_net on clean & G-adv"):
        imgs_norm = imgs_norm.to(device)
        labels = labels.to(device)
        B = imgs_norm.size(0)
        total += B

        # Clean preds for T_net
        preds_T_clean = T_net(imgs_norm).argmax(dim=1)
        correct_clean_T += (preds_T_clean == labels).sum().item()

        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        correct_clean_R += (preds_R_clean == labels).sum().item()

        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()

        # Build adversarial images using the trained G
        imgs_pixel = denormalize(imgs_norm)               # normalized -> pixel [0,1]
        delta = G(imgs_pixel)                             # G output (pixel-space)
        delta = torch.clamp(delta, -eps, eps)             # clamp in pixel-space
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)                 # back to normalized input for networks

        # Evaluate T_net on adv images
        preds_T_adv = T_net(adv_norm).argmax(dim=1)
        correct_adv_T += (preds_T_adv == labels).sum().item()


        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        correct_adv_R += (preds_R_adv == labels).sum().item()

        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        correct_adv_f += (preds_f_adv == labels).sum().item()

# compute percentages
clean_acc_T = 100.0 * correct_clean_T / total
adv_acc_T   = 100.0 * correct_adv_T   / total

clean_acc_R = 100.0 * correct_clean_R / total
adv_acc_R   = 100.0 * correct_adv_R   / total

clean_acc_f = 100.0 * correct_clean_f / total
adv_acc_f   = 100.0 * correct_adv_f   / total

print("=== Final evaluation using trained G ===")
print(f"Total test samples: {total}")
print("")
print(f"T_net - clean accuracy: {clean_acc_T:.2f}%")
print(f"T_net - G-adv accuracy : {adv_acc_T:.2f}% (drop {clean_acc_T - adv_acc_T:.2f} pp)")
print("")

print(f"R_net - clean accuracy: {clean_acc_R:.2f}%")
print(f"R_net - G-adv accuracy : {adv_acc_R:.2f}% (drop {clean_acc_R - adv_acc_R:.2f} pp)")
print("")
print(f"f_net - clean accuracy: {clean_acc_f:.2f}%")
print(f"f_net - G-adv accuracy : {adv_acc_f:.2f}% (drop {clean_acc_f - adv_acc_f:.2f} pp)")


In [ ]:
#================================================================
# In the The Following Cells:
# The accuracy of R-net and T-net against untargeted transferable attacks crafted on f-net (C-net) using FGSM, PGD, and C&W attacks is evaluated.
# ===============================================================

In [ ]:

# ===============================================
# In This Cell:
# Untargeted FGSM attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted FGSM attacks crafted on f_net.
# ===============================================


import torch
from tqdm import tqdm
import torch.nn.functional as F

# ensure eval mode
f_net.eval()
R_net.eval()
T_net.eval()

# Pixel-space epsilon
eps_pixel = 8.0 / 255.0

# CIFAR normalization 
mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
std  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# normalized-space epsilon and clamps
eps_norm = (eps_pixel / std).to(device)
min_norm = ((0.0 - mean) / std).to(device)
max_norm = ((1.0 - mean) / std).to(device)

criterion = torch.nn.CrossEntropyLoss(reduction='mean')

# counters
total = 0

# clean accuracy counters
correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

# adversarial (FGSM-from-f) accuracy counters
correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# transfer stats: among samples that fool f_net, how many also fool R_net / T_net
f_fool_count = 0
transfer_R_on_f_fool = 0
transfer_T_on_f_fool = 0


for imgs_norm, labels in tqdm(test_dataloader, desc="FGSM eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B

    # --- Clean predictions ---
    with torch.no_grad():
        logits_f_clean = f_net(imgs_norm)
        preds_f_clean = logits_f_clean.argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()

        logits_R_clean = R_net(imgs_norm)
        preds_R_clean = logits_R_clean.argmax(dim=1)
        correct_clean_R += (preds_R_clean == labels).sum().item()

        logits_T_clean = T_net(imgs_norm)
        preds_T_clean = logits_T_clean.argmax(dim=1)
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # --- Craft FGSM adversarial examples w.r.t. f_net (normalized-space) ---
    imgs_norm_adv = imgs_norm.clone().detach().requires_grad_(True)

    logits = f_net(imgs_norm_adv)
    loss = criterion(logits, labels)
    loss.backward()

    # sign gradient in normalized space
    grad_sign = imgs_norm_adv.grad.data.sign()

    # build normalized delta corresponding to eps_pixel per channel
    delta_norm = eps_norm * grad_sign

    adv_norm = torch.clamp(imgs_norm_adv + delta_norm, min_norm, max_norm)

    # --- Evaluate all three networks on adv_norm  ---
    with torch.no_grad():
        logits_f_adv = f_net(adv_norm)
        preds_f_adv = logits_f_adv.argmax(dim=1)
        correct_adv_f += (preds_f_adv == labels).sum().item()

        logits_R_adv = R_net(adv_norm)
        preds_R_adv = logits_R_adv.argmax(dim=1)
        correct_adv_R += (preds_R_adv == labels).sum().item()

        logits_T_adv = T_net(adv_norm)
        preds_T_adv = logits_T_adv.argmax(dim=1)
        correct_adv_T += (preds_T_adv == labels).sum().item()

    # --- Transfer stats: among samples where f_net was fooled, how many are also fooled in R/T ---
    f_fool_mask = (preds_f_adv != labels)
    n_f_fool = f_fool_mask.sum().item()
    if n_f_fool > 0:
        f_fool_count += n_f_fool
        # for those indices, count R_net misclassification
        transfer_R_on_f_fool += ((preds_R_adv != labels) & f_fool_mask).sum().item()
        transfer_T_on_f_fool += ((preds_T_adv != labels) & f_fool_mask).sum().item()

# --- final metrics ---
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f   = 100.0 * correct_adv_f   / total
adv_acc_R   = 100.0 * correct_adv_R   / total
adv_acc_T   = 100.0 * correct_adv_T   / total

# Transfer rates
transfer_rate_R = 100.0 * transfer_R_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')
transfer_rate_T = 100.0 * transfer_T_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')

print("\n=== FGSM (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")
print(f"\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (FGSM eps={eps_pixel:.5f}) accuracy (models evaluated on adv examples crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nFGSM attack success on f_net (f_net fooled count): {f_fool_count} / {total}  ({100.0 * f_fool_count/total:.2f}%)")
print(f"Of those f_net-fooled examples, transfer rate -> R_net: {transfer_rate_R:.2f}%  |  T_net: {transfer_rate_T:.2f}%")


In [ ]:

# ===============================================
# In This Cell:
# Untargeted PGD attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted PGD attacks crafted on f_net.
# ===============================================

import torch
from tqdm import tqdm
import torch.nn.functional as F

# Ensure models in eval mode
f_net.eval()
R_net.eval()
T_net.eval()


eps_pixel = 4.0 / 255.0          # same as your FGSM
pgd_steps = 30                   # number of PGD iterations (tuneable)
alpha_pixel = 2.0 / 255.0        # step per iteration (pixel-space)
random_start = True              # random start inside eps-ball

# CIFAR normalization
mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
std  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# normalized-space equivalents
eps_norm = (eps_pixel / std).to(device)
alpha_norm = (alpha_pixel / std).to(device)
min_norm = ((0.0 - mean) / std).to(device)
max_norm = ((1.0 - mean) / std).to(device)

criterion = torch.nn.CrossEntropyLoss(reduction='mean')

# Counters
total = 0
correct_clean_f = correct_clean_R = correct_clean_T = 0
correct_adv_f = correct_adv_R = correct_adv_T = 0

# Transfer stats: among samples that fool f_net, how many also fool R_net / T_net
f_fool_count = 0
transfer_R_on_f_fool = 0
transfer_T_on_f_fool = 0

print(f"Running PGD-{pgd_steps} (eps={eps_pixel:.5f}, alpha={alpha_pixel:.5f}) crafted on f_net ...")

for imgs_norm, labels in tqdm(test_dataloader, desc=f"PGD-{pgd_steps} eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B

    # Clean predictions 
    with torch.no_grad():
        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        preds_T_clean = T_net(imgs_norm).argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # --- PGD attack crafted on f_net (work in normalized space) ---
    # Start adv (normalized)
    if random_start:
        # random uniform in pixel space in [-eps_pixel, eps_pixel], convert to normalized delta
        rand_delta_pixel = (torch.rand_like(imgs_norm) * 2.0 - 1.0) * eps_pixel
        rand_delta_norm = rand_delta_pixel / std
        adv = torch.clamp(imgs_norm + rand_delta_norm, min_norm, max_norm).detach()
    else:
        adv = imgs_norm.clone().detach()

    adv.requires_grad_(True)

    for _step in range(pgd_steps):
        logits = f_net(adv)
        loss = criterion(logits, labels)
        # compute gradient wrt adv
        loss.backward()
        grad_sign = adv.grad.data.sign()
        # step in normalized space
        adv = adv.detach() + alpha_norm * grad_sign
        # project back to the L-inf ball around original imgs_norm (in normalized space)
        adv = torch.max(torch.min(adv, imgs_norm + eps_norm), imgs_norm - eps_norm)
        # clamp to valid normalized image range
        adv = torch.clamp(adv, min_norm, max_norm).detach()
        adv.requires_grad_(True)

    adv_norm = adv.detach()

    # Evaluate all three nets on adv_norm 
    with torch.no_grad():
        preds_f_adv = f_net(adv_norm).argmax(dim=1)
        preds_R_adv = R_net(adv_norm).argmax(dim=1)
        preds_T_adv = T_net(adv_norm).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

    # transfer statistics: among samples where f_net was fooled, how many are also fooled in R/T
    f_fool_mask = (preds_f_adv != labels)
    n_f_fool = f_fool_mask.sum().item()
    if n_f_fool > 0:
        f_fool_count += n_f_fool
        transfer_R_on_f_fool += ((preds_R_adv != labels) & f_fool_mask).sum().item()
        transfer_T_on_f_fool += ((preds_T_adv != labels) & f_fool_mask).sum().item()

# Final metrics
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f   = 100.0 * correct_adv_f   / total
adv_acc_R   = 100.0 * correct_adv_R   / total
adv_acc_T   = 100.0 * correct_adv_T   / total

transfer_rate_R = 100.0 * transfer_R_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')
transfer_rate_T = 100.0 * transfer_T_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')

print("\n=== PGD (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")
print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (PGD eps={eps_pixel:.5f}, steps={pgd_steps}) accuracy (models evaluated on adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nPGD fooled count on f_net: {f_fool_count} / {total}  ({100.0 * f_fool_count/total:.2f}%)")
print(f"Of those f_net-fooled examples, transfer rate -> R_net: {transfer_rate_R:.2f}%  |  T_net: {transfer_rate_T:.2f}%")


In [ ]:


# ===============================================
# In This Cell:
# Untargeted C&W attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted C&W attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

# ensure eval mode
f_net.eval()
R_net.eval()
T_net.eval()

# CIFAR normalization
mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1,3,1,1)
std  = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1,3,1,1)

# C&W hyperparameters (tune these)
cw_steps = 1000        # optimization steps per batch


c = 1e-1              # weight for L2 penalty term
kappa = 0.0           # confidence margin
early_stop = True     # stop if all samples in batch are already misclassified by f_net
verbose_batch = False # print per-batch debug info if True

criterion = torch.nn.CrossEntropyLoss(reduction='none')

# Counters
total = 0
correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# Transfer stats among f_net-fooled samples
f_fool_count = 0
transfer_R_on_f_fool = 0
transfer_T_on_f_fool = 0

print(f"Starting C&W-style L2 attack on f_net: steps={cw_steps}, lr={lr_cw}, c={c}, kappa={kappa}")

for imgs_norm, labels in tqdm(test_dataloader, desc="C&W eval (f->transfer)"):
    imgs_norm = imgs_norm.to(device)
    labels = labels.to(device)
    B = imgs_norm.size(0)
    total += B

    # Compute clean preds first 
    with torch.no_grad():
        preds_f_clean = f_net(imgs_norm).argmax(dim=1)
        preds_R_clean = R_net(imgs_norm).argmax(dim=1)
        preds_T_clean = T_net(imgs_norm).argmax(dim=1)
        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # Work in pixel space for delta optimization:
    imgs_pixel = denormalize(imgs_norm)   # [0,1] pixel space

    # Initialize delta (pixel-space), start from zero

    delta = torch.zeros_like(imgs_pixel, requires_grad=True, device=device)

    opt_delta = torch.optim.Adam([delta], lr=lr_cw)

    # Precompute true logits for logging, but optimization uses current logits
    target_labels = labels

    with torch.no_grad():
        logits_f_clean = f_net(imgs_norm)
        clean_correct_mask = (logits_f_clean.argmax(dim=1) == labels)

    # C&W optimization loop
    for step in range(cw_steps):
        opt_delta.zero_grad()

        # build adv pixel and clamp to valid range [0,1]
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)  # normalized input for models

        logits_f = f_net(adv_norm)   # shape [B, C]

        # For untargeted C&W: want true logit <= max(other) - kappa
        true_logits = logits_f[torch.arange(B), labels]
        other_logits = logits_f.clone()
        other_logits[torch.arange(B), labels] = -1e9
        max_other, _ = other_logits.max(dim=1)
        # hinge term: max(true - max_other + kappa, 0)
        margin = true_logits - max_other + kappa
        hinge = torch.clamp(margin, min=0.0)

        loss_hinge = hinge.mean()
        loss_l2 = (delta.view(B, -1).pow(2).sum(dim=1)).mean()
        loss = c * loss_l2 + loss_hinge

        loss.backward()
        opt_delta.step()


        # Early stopping: if all samples in batch are already misclassified by f_net (hinge==0)
        if early_stop:
            with torch.no_grad():
                logits_f_now = logits_f
                preds_f_now = logits_f_now.argmax(dim=1)
                # if all misclassified (i.e., preds != true labels) then break
                if (preds_f_now != labels).all():
                    break

    # After optimization, evaluate final adv
    with torch.no_grad():
        adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
        adv_norm = renormalize(adv_pixel)

        logits_f_adv = f_net(adv_norm)
        preds_f_adv = logits_f_adv.argmax(dim=1)
        correct_adv_f += (preds_f_adv == labels).sum().item()

        logits_R_adv = R_net(adv_norm)
        preds_R_adv = logits_R_adv.argmax(dim=1)
        correct_adv_R += (preds_R_adv == labels).sum().item()

        logits_T_adv = T_net(adv_norm)
        preds_T_adv = logits_T_adv.argmax(dim=1)
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # For transfer stats: among samples where f_net was fooled, how many are also fooled in R/T
        f_fool_mask = (preds_f_adv != labels)
        n_f_fool = f_fool_mask.sum().item()
        if n_f_fool > 0:
            f_fool_count += n_f_fool
            transfer_R_on_f_fool += ((preds_R_adv != labels) & f_fool_mask).sum().item()
            transfer_T_on_f_fool += ((preds_T_adv != labels) & f_fool_mask).sum().item()

    if verbose_batch:
        print(f"Batch done: clean_correct_f {(preds_f_clean==labels).sum().item()}  fooled_f {n_f_fool}")

# Final aggregated metrics
clean_acc_f = 100.0 * correct_clean_f / total
clean_acc_R = 100.0 * correct_clean_R / total
clean_acc_T = 100.0 * correct_clean_T / total

adv_acc_f   = 100.0 * correct_adv_f / total
adv_acc_R   = 100.0 * correct_adv_R / total
adv_acc_T   = 100.0 * correct_adv_T / total

transfer_rate_R = 100.0 * transfer_R_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')
transfer_rate_T = 100.0 * transfer_T_on_f_fool / f_fool_count if f_fool_count > 0 else float('nan')

print("\n=== C&W-style (untargeted) attack crafted on f_net (batch-optimized) ===")
print(f"Total test samples: {total}")
print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (C&W-style) accuracy (models evaluated on adv examples crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nC&W attack success on f_net (fooled count): {f_fool_count} / {total}  ({100.0 * f_fool_count/total:.2f}%)")
print(f"Of those f_net-fooled examples, transfer rate -> R_net: {transfer_rate_R:.2f}%  |  T_net: {transfer_rate_T:.2f}%")


In [ ]:

# ===============================================
# In This Cell:
# Visualization Cell: Display 10 clean test images with their predictions, 10 adversarial images generated by G with their predictions, and the corresponding differences

# ===============================================


import matplotlib.pyplot as plt
import torch


f_net.eval()
R_net.eval()
G.eval()

# get one batch from test set
dataiter = iter(test_dataloader)
imgs_norm, labels = next(dataiter)
# ensure at least 10 available in the batch
n_show = min(10, imgs_norm.size(0))
imgs_norm = imgs_norm[:n_show].to(device)
labels = labels[:n_show].to(device)

# Convert normalized -> pixel [0,1] for generator
imgs_pixel = denormalize(imgs_norm)      # [0,1]

with torch.no_grad():
    # Generate perturbation in pixel space, clamp to eps and valid range
    delta = G(imgs_pixel)                        # G outputs in [-1,1] (Tanh)
    delta = torch.clamp(delta, -eps, eps)        # bound in pixel space
    adv_pixel = torch.clamp(imgs_pixel + delta, 0.0, 1.0)
    adv_norm = renormalize(adv_pixel)            # back to normalized space for models

    # Predictions on clean (normalized) and adv (normalized)
    preds_f_clean = f_net(imgs_norm).argmax(dim=1)
    preds_R_clean = R_net(imgs_norm).argmax(dim=1)
    preds_f_adv   = f_net(adv_norm).argmax(dim=1)
    preds_R_adv   = R_net(adv_norm).argmax(dim=1)

# Plot: 4 rows x n_show columns
plt.figure(figsize=(3*n_show, 10))

for i in range(n_show):
    # Row 1: Clean image (pixel space)
    ax = plt.subplot(4, n_show, 1 + i)
    img = imgs_pixel[i].cpu().permute(1,2,0).numpy()
    plt.imshow(img)
    ax.set_title(f"T:{labels[i].item()}\nR:{preds_R_clean[i].item()} f:{preds_f_clean[i].item()}", fontsize=9)
    ax.axis('off')

    ax = plt.subplot(4, n_show, n_show + 1 + i)
    adv_img = adv_pixel[i].cpu().permute(1,2,0).numpy()
    plt.imshow(adv_img)
    ax.set_title(f"R:{preds_R_adv[i].item()} f:{preds_f_adv[i].item()}", fontsize=9)
    ax.axis('off')

    # Row 3: Difference (grayscale) = adv - clean averaged over channels
    ax = plt.subplot(4, n_show, 2*n_show + 1 + i)
    diff = (adv_pixel[i] - imgs_pixel[i]).cpu().permute(1,2,0).mean(axis=2)
    plt.imshow(diff, cmap='gray')
    ax.set_title("Gray Δ", fontsize=9)
    ax.axis('off')

    # Row 4: Difference 
    ax = plt.subplot(4, n_show, 3*n_show + 1 + i)
    diff = (adv_pixel[i] - imgs_pixel[i]).cpu().permute(1,2,0).mean(axis=2)
    plt.imshow(diff, cmap='seismic', vmin=-eps, vmax=eps)
    ax.set_title("Seismic Δ", fontsize=9)
    ax.axis('off')

plt.suptitle("Row1: Clean (T/ R / f) | Row2: Adv (R / f) | Row3: Δ (gray) | Row4: Δ (seismic)", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()
